# **Compressible Flow Simulation**

## Learning objectives <a id='learningObjectives'></a>

Welcome to the tutorial on the `Compressible Flow simulation` in Nektar++ framework. This tutorial aims to show how to setup and run a simulation for inviscid compressible flows (governed by Euler equations) using a well known geometry of NACA0012 airfoil. 

After completetion of this tutorial you will be familiar with: 
- Configuring a session file suitable for compressible Euler equations in Nektar++
- Solving the compressible flows explicitly and implicitly

<div class="alert alert-warning">

**Note:** It is suggested to download the cases to your local machine or cluster to complete the tutorial. 
    
</div>

# Introduction <a id='introduction'></a>
In this tutorial we walk through the user how to configure a session file for solving the compressible Euler equations using a simple flow example. We consider the 2D flow over the NACA0012 airfoil.

## Governing equation <a id='geqn' ></a>

The compressible solver implemented in ***Nektar++*** uses the conservative form of the governing equations.

For Navier-Stokes equations:
\begin{equation}
  \label{eq::NS}
  \frac{\partial \mathbf{Q}}{\partial t} + 
  \frac{\partial \mathbf{F}^c_1}{\partial x_1} + 
  \frac{\partial \mathbf{F}^c_2}{\partial x_2} + 
  \frac{\partial \mathbf{F}^c_3}{\partial x_3} = 
  \frac{\partial \mathbf{F}^v_1}{\partial x_1} + 
  \frac{\partial \mathbf{F}^v_2}{\partial x_2} + 
  \frac{\partial \mathbf{F}^v_3}{\partial x_3}
\end{equation}

For Euler equations:
\begin{equation}
  \label{eq::Euler}
  \frac{\partial \mathbf{Q}}{\partial t} + 
  \frac{\partial \mathbf{F}^c_1}{\partial x_1} + 
  \frac{\partial \mathbf{F}^c_2}{\partial x_2} + 
  \frac{\partial \mathbf{F}^c_3}{\partial x_3} = 0
\end{equation}

In the above $x_i$ denotes the $x$, $y$, and $z$-directions for $i=1,2,3$, respectively, $\mathbf{Q}$ is the vector of conservative variables, $\mathbf{F}^c_i = \mathbf{F}^c_i(\mathbf{Q})$ and $\mathbf{F}^v_i = \mathbf{F}^v_i(\mathbf{Q},\nabla \mathbf{Q})$ are vectors for convective flux and viscous flux, respectively
\begin{equation}
\label{eq::NS_vectors}
  \mathbf{Q} =\left(
    \begin{array}{c}
      \rho \\ \rho u \\ \rho v \\ \rho w \\ E
    \end{array} \right), \hspace{0.5 cm}
  \mathbf{F}^c_i =\left(
    \begin{array}{c}
      \rho u_i \\ \rho u_1 u_i + p \delta_{1i} \\ \rho u_2 u_i + p \delta_{2i} \\ \rho u_3 u_i + p \delta_{3i} \\ u_i (E + p)
    \end{array} \right), \hspace{0.5 cm}
  \mathbf{F}^v_i =\left(
    \begin{array}{c}
      0 \\ \tau_{1i} \\ \tau_{2i} \\ \tau_{3i} \\ u_j\tau_{ji} + \kappa \partial T / \partial x_i
    \end{array} \right)
\end{equation}
where $\rho$ is the density, $u_i$ is the velocity component in the $x_i$-direction, $p$ is the pressure, $\delta_{ij}$ is the Kronecker delta function, and $E$ is the total energy per unit volume. As for the viscous flux vectors, $T$ is the temperature, $\kappa$ is the thermal conductivity, $\tau_{ij}$ is stress tensor component.

# Problem description  <a id='pdes' ></a>

In this tutorial we simulate a 2D compressible flow over NACA0012 airfoil. The simulation is non-dimesionalized so that the chord length is scaled to a unit, $L=1.0$, and the following freestream parameters are used:

| $$\gamma$$ |$$Ma$$ | $$R$$ | $$T_{\infty}$$ | $$\rho_{\infty}$$ | $$U_{\infty}$$ | $$\alpha$$ |
| --- | --- | --- | --- | --- | --- | --- |
| $$1.4$$ | $$0.4$$ | $$\frac{1}{\gamma Ma^2}$$ | $$1.0$$ | $$1.0$$ | $$1.0$$ | $$0.0$$ |


The flow domain is a rectangle of sizes $[-5,10] \times [-5,5]$ as shown in the figure below. 

<table class="center"> 
<tr> 
    <td> <img src="mesh_full.png" width="400" height ="100"/>  </td> 
    <td> <img src="mesh_close.png" width="400" height = "100"/> </td>
    
</tr> 
<tr>
    <td style="text-align:center"> Full domain and mesh </td>
    <td style="text-align:center"> Near wall mesh  </td>  
</tr> 
</table>
    
We apply the following boundary conditions (BCs): 

- No-penetration condition on the airfoil surface;
- Riemann boundary condition at the boundary of the other four boundaries (inflow, outflow, top, and bottom) 


<div class="alert alert-info">

**Note:** For the top and bottom boundaries, a periodic boundary condition in the $y$-direction can be used instead.

</div>


# Session file setup: explicit Euler<a id='sessionFile_explicit_Euler'></a>

In this section we go through the settings in the session file (**.xml**) for explicit simulations. Please open the session file [**session_naca_ex.xml**](session_naca_ex.xml). First, let us complete the `<EXPANSIONS>` tags.

## Expansion bases <a id='expansionBases'></a>

**Task 1:** Please write the missing parts in `<EXPANSIONS>` tag. A completed `<EXPANSIONS>` tag has the following standard form:
    
```xml
  <EXPANSIONS>
    <E COMPOSITE="C[1]" NUMMODES="2" FIELDS="rho,rhou,rhov,E" TYPE="MODIFIED" /> <!--quad elements-->
    <E COMPOSITE="C[2]" NUMMODES="2" FIELDS="rho,rhou,rhov,E" TYPE="MODIFIED" /> <!--tri elements-->
  </EXPANSIONS>`
 ```
 
 Here, 
- `COMPOSITE` refers to the ID assigned to the geometric domain (or sub-domain) defined in the geometry xml file. 
- `NUMMODES` is used to indentify the polynomial order of the expansion basis functions as P = `NUMMODES`-1.
- `TYPE` denotes the basis functions. Possible choices are `MODIFIED`, `GLL_LAGRANGE`,`GLL_LAGRANGE_SEM`. One may note that the first basis type uses modal expansion whereas the remaining two consider nodal expansion.  


## Conditions setup <a id='conditionsSetup'></a>

Now, we will complete the `<CONDITIONS>` tag. There are **five** sections in this tag:

- SolverInfo
- Parameters
- Variables
- Boundary Conditions
- Initial Conditions


### Solver info <a id='solverInfo'></a>

**Task 2:** Please complete the properties `EQTYPE`,  `TimeIntegrationMethod`, and `UpwindType` as shown below, 

```xml
      <I PROPERTY="EQType"                VALUE="EulerCFE"            />
      <I PROPERTY="TimeIntegrationMethod" VALUE="RungeKutta2_SSP"     />
      <I PROPERTY="UpwindType"            VALUE="Roe"                 />
```
where
- `EQTYPE` sets the kind of equations we want to solve. Here we use the Euler equations;
- `TimeIntegrationMethod` sets the integration method for time advancing. Here we use the 2nd order Runge-Kutta; 
- `UpwindType` sets the Riemann solver numerical inviscid flux computation. Here we use the approximate Riemann solver of Roe.


### Parameters  <a id='parameters'></a>
In `<PARAMETERS>` tag, we can define the parameters present in the flow problem as well as the parameters required by various solvers of Nektar++. Hence a parameter can either be defined in the context of the problem we are solving (dummay variable), or be predefined within Nektar++ (solver parameter). One should note that the names of the solver parameters can't be altered. 

**Task 3:** Assign values to the following parameters as  

```xml
      <P> Ma                    = 0.4                     </P>
      <P> GasConstant           = 1.0/(Gamma*Ma^2)        </P>
      <P> AoA                   = 0.0/180.0*PI            </P>
      <P> Timestep              = 5.0e-5                  </P>
      <P> NumSteps              = 40000                   </P>    
```
Here,
- `Ma` is the Mach number. It is not necessary but set to compute non-dimensionalized `GasConstant` conveniently;
- `GasConstant` is the gas constant predefined in Nektar++.
- `TimeStep` sets the time-step size for the time integration. One should select magnitude of `TimeStep` strategically to avoid the situation of CFL (Courant-Friedrichs-Lewy) number to be too large. However, in the following implicit session file, this value can be much larger!
- `NumSteps` sets the number of time-steps required for the simulation.

### Variables  <a id='variables'></a>

In `<VARIABLES>` tag, we define the number (and name) of solution variables. We prescribed a unique ID to each variable. 

**Task 4:** Provide a ID for the conservative variable $\rho v$ as 
```xml
      <V ID="2"> rhov </V>
```

### Boundary conditions  <a id='bc'></a>

Boundary conditions are defined by two tags. First we define the boundary regions in the `<BOUNDARYREGIONS>` tag in terms of composite `C[ ]` entities from the `<GEOMETRY>` tag section of the [mesh xml](cyl2DMesh.xml) file. Each boundary region has a unique ID. For the current computational domain there are seven composites (see towards the bottom section of [naca.xml](naca.xml) file) marked as C[1] to C[7].  

**Task 5:** Define the inflow region using C[4] as shown below

```xml
      <B ID="3"> C[4] </B>   <!-- inflow -->
```

The next tag we define is `<BOUNDARYCONDITIONS>` tag by which we actually specify the boundary conditions for each boundary ID specified in the above`<BOUNDARYREGIONS>` tag.

**Task 6:** Define the the conservative variables at inflow as the farfield quantities as 
```xml       
      <REGION REF="3">
        <D VAR="rho"    VALUE="rhoInf"     />
        <D VAR="rhou"   VALUE="rhoInf * uInf"   />
        <D VAR="rhov"   VALUE="rhoInf * vInf"   />
        <D VAR="E"      VALUE="pInf / (Gamma - 1) + 0.5 * rhoInf * (uInf * uInf + vInf * vInf)"/>
      </REGION>
           
```




### Initial conditions  <a id='ic'></a>

**Task 7:** Provide the initial conditions as the freestream conditions in the "InitialConditions" `FUNCTION` as

```xml
        <E VAR="rho"    VALUE="rhoInf"     />
        <E VAR="rhou"   VALUE="rhoInf * uInf"   />
        <E VAR="rhov"   VALUE="rhoInf * vInf"   />
        <E VAR="E"      VALUE="pInf / (Gamma - 1) + 0.5 * rhoInf * (uInf * uInf + vInf * vInf)"/>
                 
```


# Running the solver <a id='basic_runningSolver'></a>

The two xml files provided in this tutorial are:
- Geometry file: [naca.xml](naca.xml)
- Condition file: [session_naca.xml](session_naca.xml)

The flow solvers in Nektar++ can be run either in serial or parallel. However, for this case it is suggessted to run in parallel on your local machine. Here we use `!` (the jupyter notebook syntax for bash command) to run the solver in this online interactive tutorial which used Docker imange and Binder to run Nektar++ in the background.

**Task 8:** Go to the next cell and click the `Run` icon to run the solver. In this example we use 4 physical cores to solve the problem.


<div class="alert alert-warning">

**Note:** Use of high number of processors might not work in the current online tutorial. Also, parallel run might be slower than the serial run due to resource constrain.

</div>

<div class="alert alert-warning">

**Note:** To run the solver in serial, remove the `mpirun -np N` part.

</div>


In [1]:
!mpirun -np 4 CompressibleFlowSolver naca.xml session_naca_ex-Complete.xml

gs_setup: 290 unique labels shared
   pairwise times (avg, min, max): 1.24138e-06 1.2205e-06 1.2558e-06
   crystal router                : 2.245e-06 2.2254e-06 2.27103e-06
   all reduce                    : 2.77909e-06 2.72729e-06 2.8288e-06
   used all_to_all method: pairwise
   handle bytes (avg, min, max): 3756 3092 4516
   buffer bytes (avg, min, max): 2320 1888 2816
gs_setup: 290 unique labels shared
   pairwise times (avg, min, max): 1.15705e-06 1.14245e-06 1.19517e-06
   crystal router                : 2.25196e-06 2.23555e-06 2.25799e-06
   all reduce                    : 2.64696e-06 2.60947e-06 2.66479e-06
   used all_to_all method: pairwise
   handle bytes (avg, min, max): 3756 3092 4516
   buffer bytes (avg, min, max): 2320 1888 2816
gs_setup: 290 unique labels shared
   pairwise times (avg, min, max): 1.26855e-06 1.23549e-06 1.29472e-06
   crystal router                : 2.22756e-06 2.19028e-06 2.25045e-06
   all reduce                    : 2.62342e-06 2.61217e-06 2.6295e-06

Steps: 10500    Time: 0.525        CPU Time: 0.350052s
Steps: 10600    Time: 0.53         CPU Time: 0.349149s
Steps: 10700    Time: 0.535        CPU Time: 0.350286s
Steps: 10800    Time: 0.54         CPU Time: 0.350235s
Steps: 10900    Time: 0.545        CPU Time: 0.349498s
Steps: 11000    Time: 0.55         CPU Time: 0.352918s
Steps: 11100    Time: 0.555        CPU Time: 0.350615s
Steps: 11200    Time: 0.56         CPU Time: 0.35082s
Steps: 11300    Time: 0.565        CPU Time: 0.352131s
Steps: 11400    Time: 0.57         CPU Time: 0.349375s
Steps: 11500    Time: 0.575        CPU Time: 0.352983s
Steps: 11600    Time: 0.58         CPU Time: 0.351309s
Steps: 11700    Time: 0.585        CPU Time: 0.350926s
Steps: 11800    Time: 0.59         CPU Time: 0.351226s
Steps: 11900    Time: 0.595        CPU Time: 0.350476s
Steps: 12000    Time: 0.6          CPU Time: 0.35233s
Steps: 12100    Time: 0.605        CPU Time: 0.350974s
Steps: 12200    Time: 0.61         CPU Time: 0.350453s
Steps: 12300

Steps: 25100    Time: 1.255        CPU Time: 0.358484s
Steps: 25200    Time: 1.26         CPU Time: 0.359304s
Steps: 25300    Time: 1.265        CPU Time: 0.359234s
Steps: 25400    Time: 1.27         CPU Time: 0.360685s
Steps: 25500    Time: 1.275        CPU Time: 0.359273s
Steps: 25600    Time: 1.28         CPU Time: 0.360199s
Steps: 25700    Time: 1.285        CPU Time: 0.359381s
Steps: 25800    Time: 1.29         CPU Time: 0.358243s
Steps: 25900    Time: 1.295        CPU Time: 0.361418s
Steps: 26000    Time: 1.3          CPU Time: 0.359699s
Steps: 26100    Time: 1.305        CPU Time: 0.359781s
Steps: 26200    Time: 1.31         CPU Time: 0.358146s
Steps: 26300    Time: 1.315        CPU Time: 0.360716s
Steps: 26400    Time: 1.32         CPU Time: 0.358729s
Steps: 26500    Time: 1.325        CPU Time: 0.358628s
Steps: 26600    Time: 1.33         CPU Time: 0.360351s
Steps: 26700    Time: 1.335        CPU Time: 0.358981s
Steps: 26800    Time: 1.34         CPU Time: 0.358519s
Steps: 269

Steps: 39800    Time: 1.99         CPU Time: 0.357758s
Steps: 39900    Time: 1.995        CPU Time: 0.360275s
Steps: 40000    Time: 2            CPU Time: 0.35823s
renaming "naca_8.chk" -> "naca_8.bak1.chk"
Writing: "naca_8.chk" (0.0158526s, XML)
Time-integration  : 142.1s
renaming "naca.fld" -> "naca.bak4.fld"
Writing: "naca.fld" (0.0158572s, XML)
-------------------------------------------
Total Computation Time = 158s
-------------------------------------------
L 2 error (variable rho) : 12.2441
L inf error (variable rho) : 1.08473
L 2 error (variable rhou) : 12.2431
L inf error (variable rhou) : 1.40827
L 2 error (variable rhov) : 0.0787936
L inf error (variable rhov) : 0.617216
L 2 error (variable E) : 142.775
L inf error (variable E) : 12.6828


# Post-processing <a id='post_processing'></a>
To visualise the flowfield first we convert the **.fld** file (generated by the solver) to a **.vtu** format file by using the `FieldConvert` ulitity available in the Nektar++. 

**Task 9:** Run the following code cell to convet the **.fld** into a **.vtu** file.

In [2]:
!FieldConvert -f naca.xml session_naca_ex-Complete.xml naca.fld flowfield_ex.vtu

Writing: "flowfield.vtu"
Written file: flowfield.vtu


<div class="alert alert-info">

**Note:** `-f` is supplied to force output, otherwise the user would be prompted whether to overwrite an existing file.

</div>

<div class="alert alert-warning">

**Note:** It is strongly recommended to use `-f` option when one is running the commands in Jupyter notebook cells.

</div>


**Task 10:** Run the following `python` script which uses  `pyvista` combined with `itkwidgets` to visualise the flowfield in-browser.

In [3]:
import pyvista as pv

# First read the VTK file
mesh = pv.read('flowfield_ex.vtu')

# Now create an itkwidget to visualise this in-browser.
pl = pv.PlotterITK()
pl.add_mesh(mesh, scalars=mesh['u'], smooth_shading=True)
pl.show(True)

Viewer(geometries=[{'vtkClass': 'vtkPolyData', 'points': {'vtkClass': 'vtkPoints', 'name': '_points', 'numberO…

Viewer(geometries=[{'vtkClass': 'vtkPolyData', 'points': {'vtkClass': 'vtkPoints', 'name': '_points', 'numberO…

<table class="center"> 
<tr> 
    <td> <img src="result_full_P4.png" width="600" height ="200"/>  </td> 
    
</tr> 
<tr>
    <td style="text-align:center"> The velocity field solving by set `NUMMODES=4` </td>
</tr> 
</table>

**********

# Session file setup: implicit Euler <a id='sessionFile_implicit_Euler'></a>

In Nektar++ the compressible flows can also be solved implicitly. The majority of the session file is the same as the one for explicit solving while the differences only sit in the `SOLVERINFO` and `PARAMETERS` tag. Please open the session file [**session_naca_im.xml**](session_naca_im.xml) and complete the following steps.

### Solver info <a id='solverInfo_im'></a>

**Task 11:** Please complete the properties `EQTYPE`,  `AdvectionAdvancement`, and `TimeIntegrationMethod` as shown below, 

```xml
      <I PROPERTY="EQType"                VALUE="EulerImplicitCFE"/>
      <I PROPERTY="AdvectionAdvancement"  VALUE="Implicit"        />
      <I PROPERTY="TimeIntegrationMethod" VALUE="DIRKOrder2"      />
```
where `EQTYPE` is set to be the implicit type of Euler equations, `AdvectionAdvancement` and `TimeIntegrationMethod` of the implicit versions are used accordingly.

### Parameters  <a id='parameters_im'></a>
Since we are now solving the Euler equations implicitly, the restrictions on time step is very much relexed. However, to start from the uniform freestream condition, we still use a smaller time step to keep convergence.

**Task 12:** Assign values to the following parameters as  

```xml
      <P> Timestep              = 1.0e-4                  </P>
      <P> NumSteps              = 100                     </P>    
```

# Running the solver: starting <a id='basic_runningSolver_im_start'></a>

**Task 13:** Go to the next cell and click the `Run` icon to run the impicit solver.

In [1]:
!mpirun -np 4 CompressibleFlowSolver naca.xml session_naca_im.xml

gs_setup: 290 unique labels shared
   pairwise times (avg, min, max): 1.24138e-06 1.2205e-06 1.2558e-06
   crystal router                : 2.245e-06 2.2254e-06 2.27103e-06
   all reduce                    : 2.77909e-06 2.72729e-06 2.8288e-06
   used all_to_all method: pairwise
   handle bytes (avg, min, max): 3756 3092 4516
   buffer bytes (avg, min, max): 2320 1888 2816
gs_setup: 290 unique labels shared
   pairwise times (avg, min, max): 1.15705e-06 1.14245e-06 1.19517e-06
   crystal router                : 2.25196e-06 2.23555e-06 2.25799e-06
   all reduce                    : 2.64696e-06 2.60947e-06 2.66479e-06
   used all_to_all method: pairwise
   handle bytes (avg, min, max): 3756 3092 4516
   buffer bytes (avg, min, max): 2320 1888 2816
gs_setup: 290 unique labels shared
   pairwise times (avg, min, max): 1.26855e-06 1.23549e-06 1.29472e-06
   crystal router                : 2.22756e-06 2.19028e-06 2.25045e-06
   all reduce                    : 2.62342e-06 2.61217e-06 2.6295e-06

Steps: 10500    Time: 0.525        CPU Time: 0.350052s
Steps: 10600    Time: 0.53         CPU Time: 0.349149s
Steps: 10700    Time: 0.535        CPU Time: 0.350286s
Steps: 10800    Time: 0.54         CPU Time: 0.350235s
Steps: 10900    Time: 0.545        CPU Time: 0.349498s
Steps: 11000    Time: 0.55         CPU Time: 0.352918s
Steps: 11100    Time: 0.555        CPU Time: 0.350615s
Steps: 11200    Time: 0.56         CPU Time: 0.35082s
Steps: 11300    Time: 0.565        CPU Time: 0.352131s
Steps: 11400    Time: 0.57         CPU Time: 0.349375s
Steps: 11500    Time: 0.575        CPU Time: 0.352983s
Steps: 11600    Time: 0.58         CPU Time: 0.351309s
Steps: 11700    Time: 0.585        CPU Time: 0.350926s
Steps: 11800    Time: 0.59         CPU Time: 0.351226s
Steps: 11900    Time: 0.595        CPU Time: 0.350476s
Steps: 12000    Time: 0.6          CPU Time: 0.35233s
Steps: 12100    Time: 0.605        CPU Time: 0.350974s
Steps: 12200    Time: 0.61         CPU Time: 0.350453s
Steps: 12300

Steps: 25100    Time: 1.255        CPU Time: 0.358484s
Steps: 25200    Time: 1.26         CPU Time: 0.359304s
Steps: 25300    Time: 1.265        CPU Time: 0.359234s
Steps: 25400    Time: 1.27         CPU Time: 0.360685s
Steps: 25500    Time: 1.275        CPU Time: 0.359273s
Steps: 25600    Time: 1.28         CPU Time: 0.360199s
Steps: 25700    Time: 1.285        CPU Time: 0.359381s
Steps: 25800    Time: 1.29         CPU Time: 0.358243s
Steps: 25900    Time: 1.295        CPU Time: 0.361418s
Steps: 26000    Time: 1.3          CPU Time: 0.359699s
Steps: 26100    Time: 1.305        CPU Time: 0.359781s
Steps: 26200    Time: 1.31         CPU Time: 0.358146s
Steps: 26300    Time: 1.315        CPU Time: 0.360716s
Steps: 26400    Time: 1.32         CPU Time: 0.358729s
Steps: 26500    Time: 1.325        CPU Time: 0.358628s
Steps: 26600    Time: 1.33         CPU Time: 0.360351s
Steps: 26700    Time: 1.335        CPU Time: 0.358981s
Steps: 26800    Time: 1.34         CPU Time: 0.358519s
Steps: 269

Steps: 39800    Time: 1.99         CPU Time: 0.357758s
Steps: 39900    Time: 1.995        CPU Time: 0.360275s
Steps: 40000    Time: 2            CPU Time: 0.35823s
renaming "naca_8.chk" -> "naca_8.bak1.chk"
Writing: "naca_8.chk" (0.0158526s, XML)
Time-integration  : 142.1s
renaming "naca.fld" -> "naca.bak4.fld"
Writing: "naca.fld" (0.0158572s, XML)
-------------------------------------------
Total Computation Time = 158s
-------------------------------------------
L 2 error (variable rho) : 12.2441
L inf error (variable rho) : 1.08473
L 2 error (variable rhou) : 12.2431
L inf error (variable rhou) : 1.40827
L 2 error (variable rhov) : 0.0787936
L inf error (variable rhov) : 0.617216
L 2 error (variable E) : 142.775
L inf error (variable E) : 12.6828


**Task 14:** Use the command in the next block to rename the output **naca.fld** file as **naca_im_start.fld**

In [ ]:
!mv naca.fld/ naca_im_start.fld/

Now we can use a much larger time step restarting from the previous result. Please follow these step to re-run the solver.

**Task 15:** Assign values to the following parameters as  

```xml
      <P> Timestep              = 2.0e-3                  </P>
      <P> NumSteps              = 1000                    </P>    
```

**Task 16:**  Set the "initialConditions" `FUNCTION` as the output of the starting

```xml
      <FUNCTION NAME="InitialConditions">
        <F VAR="rho,rhou,rhov,E"    FILE="naca_im_start.fld"/>
      </FUNCTION>           
```

**Task 17:** Go to the next cell and click the `Run` icon to restart the impicit solver.

In [5]:
!mpirun -np 4 CompressibleFlowSolver naca.xml session_naca_im.xml

gs_setup: 290 unique labels shared
   pairwise times (avg, min, max): 1.24732e-06 1.215e-06 1.27451e-06
   crystal router                : 2.30079e-06 2.26684e-06 2.32877e-06
   all reduce                    : 2.83667e-06 2.79583e-06 2.86503e-06
   used all_to_all method: pairwise
   handle bytes (avg, min, max): 3756 3092 4516
   buffer bytes (avg, min, max): 2320 1888 2816
gs_setup: 290 unique labels shared
   pairwise times (avg, min, max): 1.30166e-06 1.27563e-06 1.32388e-06
   crystal router                : 2.29748e-06 2.26544e-06 2.33203e-06
   all reduce                    : 2.64156e-06 2.61944e-06 2.65921e-06
   used all_to_all method: pairwise
   handle bytes (avg, min, max): 3756 3092 4516
   buffer bytes (avg, min, max): 2320 1888 2816
gs_setup: 290 unique labels shared
   pairwise times (avg, min, max): 1.20776e-06 1.17887e-06 1.22907e-06
   crystal router                : 2.32121e-06 2.30866e-06 2.33734e-06
   all reduce                    : 2.6498e-06 2.63294e-06 2.6634e

Steps: 105      Time: 0.22         CPU Time: 0.0662842s
Steps: 106      Time: 0.222        CPU Time: 0.0664857s
Steps: 107      Time: 0.224        CPU Time: 0.066252s
Steps: 108      Time: 0.226        CPU Time: 0.0661335s
Steps: 109      Time: 0.228        CPU Time: 0.0664098s
Steps: 110      Time: 0.23         CPU Time: 0.0661458s
Steps: 111      Time: 0.232        CPU Time: 0.0662417s
Steps: 112      Time: 0.234        CPU Time: 0.0665046s
Steps: 113      Time: 0.236        CPU Time: 0.0660551s
Steps: 114      Time: 0.238        CPU Time: 0.0664189s
Steps: 115      Time: 0.24         CPU Time: 0.0964345s
Steps: 116      Time: 0.242        CPU Time: 0.12675s
Steps: 117      Time: 0.244        CPU Time: 0.126418s
Steps: 118      Time: 0.246        CPU Time: 0.223103s
Steps: 119      Time: 0.248        CPU Time: 0.126063s
Steps: 120      Time: 0.25         CPU Time: 0.126818s
Steps: 121      Time: 0.252        CPU Time: 0.127748s
Steps: 122      Time: 0.254        CPU Time: 0.127612s
S

Steps: 253      Time: 0.516        CPU Time: 0.067241s
Steps: 254      Time: 0.518        CPU Time: 0.06717s
Steps: 255      Time: 0.52         CPU Time: 0.0669337s
Steps: 256      Time: 0.522        CPU Time: 0.0670124s
Steps: 257      Time: 0.524        CPU Time: 0.0672434s
Steps: 258      Time: 0.526        CPU Time: 0.0673057s
Steps: 259      Time: 0.528        CPU Time: 0.067068s
Steps: 260      Time: 0.53         CPU Time: 0.0671737s
Steps: 261      Time: 0.532        CPU Time: 0.0669115s
Steps: 262      Time: 0.534        CPU Time: 0.165073s
Steps: 263      Time: 0.536        CPU Time: 0.0659161s
Steps: 264      Time: 0.538        CPU Time: 0.0660812s
Steps: 265      Time: 0.54         CPU Time: 0.0659327s
Steps: 266      Time: 0.542        CPU Time: 0.0658781s
Steps: 267      Time: 0.544        CPU Time: 0.0660551s
Steps: 268      Time: 0.546        CPU Time: 0.0663633s
Steps: 269      Time: 0.548        CPU Time: 0.0660642s
Steps: 270      Time: 0.55         CPU Time: 0.066016

Steps: 400      Time: 0.81         CPU Time: 0.0659615s
Steps: 401      Time: 0.812        CPU Time: 0.0656353s
Steps: 402      Time: 0.814        CPU Time: 0.0658349s
Steps: 403      Time: 0.816        CPU Time: 0.0658084s
Steps: 404      Time: 0.818        CPU Time: 0.0656171s
Steps: 405      Time: 0.82         CPU Time: 0.0656387s
Steps: 406      Time: 0.822        CPU Time: 0.0661054s
Steps: 407      Time: 0.824        CPU Time: 0.0658918s
Steps: 408      Time: 0.826        CPU Time: 0.0655149s
Steps: 409      Time: 0.828        CPU Time: 0.0659655s
Steps: 410      Time: 0.83         CPU Time: 0.0657614s
Steps: 411      Time: 0.832        CPU Time: 0.0659851s
Steps: 412      Time: 0.834        CPU Time: 0.0654777s
Steps: 413      Time: 0.836        CPU Time: 0.0658393s
Steps: 414      Time: 0.838        CPU Time: 0.0657104s
Steps: 415      Time: 0.84         CPU Time: 0.0661591s
Steps: 416      Time: 0.842        CPU Time: 0.0660561s
Steps: 417      Time: 0.844        CPU Time: 0.0

Steps: 547      Time: 1.104        CPU Time: 0.0671936s
Steps: 548      Time: 1.106        CPU Time: 0.0668957s
Steps: 549      Time: 1.108        CPU Time: 0.0672187s
Steps: 550      Time: 1.11         CPU Time: 0.0670799s
Steps: 551      Time: 1.112        CPU Time: 0.0670895s
Steps: 552      Time: 1.114        CPU Time: 0.0669561s
Steps: 553      Time: 1.116        CPU Time: 0.0671826s
Steps: 554      Time: 1.118        CPU Time: 0.0670828s
Steps: 555      Time: 1.12         CPU Time: 0.0669988s
Steps: 556      Time: 1.122        CPU Time: 0.0671238s
Steps: 557      Time: 1.124        CPU Time: 0.0671433s
Steps: 558      Time: 1.126        CPU Time: 0.0670889s
Steps: 559      Time: 1.128        CPU Time: 0.0671682s
Steps: 560      Time: 1.13         CPU Time: 0.0669582s
Steps: 561      Time: 1.132        CPU Time: 0.0669144s
Steps: 562      Time: 1.134        CPU Time: 0.0671622s
Steps: 563      Time: 1.136        CPU Time: 0.0666737s
Steps: 564      Time: 1.138        CPU Time: 0.0

Steps: 694      Time: 1.398        CPU Time: 0.0686284s
Steps: 695      Time: 1.4          CPU Time: 0.0687474s
Steps: 696      Time: 1.402        CPU Time: 0.068658s
Steps: 697      Time: 1.404        CPU Time: 0.0683176s
Steps: 698      Time: 1.406        CPU Time: 0.0686589s
Steps: 699      Time: 1.408        CPU Time: 0.0689339s
Steps: 700      Time: 1.41         CPU Time: 0.068528s
Steps: 701      Time: 1.412        CPU Time: 0.0683074s
Steps: 702      Time: 1.414        CPU Time: 0.06885s
Steps: 703      Time: 1.416        CPU Time: 0.0687522s
Steps: 704      Time: 1.418        CPU Time: 0.0695425s
Steps: 705      Time: 1.42         CPU Time: 0.0685287s
Steps: 706      Time: 1.422        CPU Time: 0.0686594s
Steps: 707      Time: 1.424        CPU Time: 0.0680259s
Steps: 708      Time: 1.426        CPU Time: 0.167506s
Steps: 709      Time: 1.428        CPU Time: 0.067164s
Steps: 710      Time: 1.43         CPU Time: 0.0678041s
Steps: 711      Time: 1.432        CPU Time: 0.0677023

Steps: 841      Time: 1.692        CPU Time: 0.0673508s
Steps: 842      Time: 1.694        CPU Time: 0.0678902s
Steps: 843      Time: 1.696        CPU Time: 0.0675893s
Steps: 844      Time: 1.698        CPU Time: 0.0674098s
Steps: 845      Time: 1.7          CPU Time: 0.0678582s
Steps: 846      Time: 1.702        CPU Time: 0.0677416s
Steps: 847      Time: 1.704        CPU Time: 0.0671056s
Steps: 848      Time: 1.706        CPU Time: 0.0680113s
Steps: 849      Time: 1.708        CPU Time: 0.0676492s
Steps: 850      Time: 1.71         CPU Time: 0.0672749s
Steps: 851      Time: 1.712        CPU Time: 0.0676177s
Steps: 852      Time: 1.714        CPU Time: 0.0677063s
Steps: 853      Time: 1.716        CPU Time: 0.0673854s
Steps: 854      Time: 1.718        CPU Time: 0.0676281s
Steps: 855      Time: 1.72         CPU Time: 0.0676262s
Steps: 856      Time: 1.722        CPU Time: 0.0673155s
Steps: 857      Time: 1.724        CPU Time: 0.0677862s
Steps: 858      Time: 1.726        CPU Time: 0.0

Steps: 989      Time: 1.988        CPU Time: 0.128626s
Steps: 990      Time: 1.99         CPU Time: 0.129501s
Steps: 991      Time: 1.992        CPU Time: 0.128947s
Steps: 992      Time: 1.994        CPU Time: 0.129196s
Steps: 993      Time: 1.996        CPU Time: 0.129379s
Steps: 994      Time: 1.998        CPU Time: 0.129037s
Steps: 995      Time: 2            CPU Time: 0.129178s
Steps: 996      Time: 2.002        CPU Time: 0.128706s
Steps: 997      Time: 2.004        CPU Time: 0.227751s
Steps: 998      Time: 2.006        CPU Time: 0.129278s
Steps: 999      Time: 2.008        CPU Time: 0.130401s
Steps: 1000     Time: 2.01         CPU Time: 0.130978s
renaming "naca_5.chk" -> "naca_5.bak0.chk"
Writing: "naca_5.chk" (0.0159675s, XML)
Time-integration  : 82.5745s
renaming "naca.fld" -> "naca.bak6.fld"
Writing: "naca.fld" (0.0158489s, XML)
-------------------------------------------
Total Computation Time = 83s
-------------------------------------------
L 2 error (variable rho) : 12.2441

# Post-processing <a id='post_processing'></a>
To visualise the flowfield first we convert the **.fld** file (generated by the solver) to a **.vtu** format file by using the `FieldConvert` ulitity available in the Nektar++. 

**Task 18:** Run the following code cell to convet the **.fld** into a **.vtu** file.

In [6]:
!FieldConvert -f naca.xml session_naca_im.xml naca.fld flowfield_im.vtu

Writing: "flowfield_im.vtu"
Written file: flowfield_im.vtu


In [8]:
import pyvista as pv

# First read the VTK file
mesh = pv.read('flowfield_im.vtu')

# Now create an itkwidget to visualise this in-browser.
pl = pv.PlotterITK()
pl.add_mesh(mesh, scalars=mesh['u'], smooth_shading=True)
pl.show(True)

Viewer(geometries=[{'vtkClass': 'vtkPolyData', 'points': {'vtkClass': 'vtkPoints', 'name': '_points', 'numberO…

Viewer(geometries=[{'vtkClass': 'vtkPolyData', 'points': {'vtkClass': 'vtkPoints', 'name': '_points', 'numberO…

## Extra tasks


**Task 19:** A better resolution can be obtained by increasing the expansion order under the `<EXPANSIONS>` tag. We can set the `NUMMODES=5` and check the result. Please noted that a higher order expansion may require the time step to be smaller to avoid divergence.  


**********

<div class="alert alert-info">

**Note:** To run the Navier-Stokes simulation, under the `<SOLVERINFO>` tag change `EulerCFE` or `EulerImplicitCFE` to be `NavierStokesCFE` or `NavierStokesImplicitCFE` for an explicit or implicit simulation, respectively . `DiffusionType`, `DiffusionAdvancement`, and `ViscosityType` need to be specified as well. Other differences include the parameters related to viscosity and heat transfer under `<PARAMETERS>` tag and the type of wall boundary condition under `<BOUNDARYCONDITIONS>` tag. An example for implicitly solving Navier-Stokes equations is provided in [**session_naca_NS-Complete.xml**](session_naca_NS-Complete.xml). More information  can also be found in the [tutorial](https://doc.nektar.info/tutorials/latest/cfs/CylinderSubsonic_NS/cfs-CylinderSubsonic_NS.pdf)

</div>

# Take-a-ways   <a id='takeaways'> </a>
It is worthy to keep in mind the following points:

- The comprssible flows can be solved either explicitly or implicitly; 
- The explicit solving requires a smaller time step to satisfy the CFL condition while the restriction is very much relexed for the implicit solving; 

# References   <a id='references'> </a>
1. Nektar++: Spectral/hp Element Framework, Version 5.0.1, [User guide](http://www.nektar.info/downloads), January, 2021.
2. Kirby R, Sherwin S. Stabilisation of spectral/hp methods through spectral vanishing viscosity: Application to fluid mechanics modelling. Comput. Methods Appl. Mech. Engrg. 2006; 195: 3128–3144

## End of the tutorial. 